In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import MACCSkeys
from modules.data_loader import load_cleaned

In [2]:
visco = load_cleaned()
visco.info()

<class 'pandas.DataFrame'>
RangeIndex: 8152 entries, 0 to 8151
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_surfactant1      8152 non-null   str    
 1   type_surfactant1    8152 non-null   str    
 2   smiles_surfactant1  8152 non-null   str    
 3   c_surfactant1       8152 non-null   float64
 4   id_surfactant2      2726 non-null   str    
 5   type_surfactant2    2726 non-null   str    
 6   smiles_surfactant2  2631 non-null   str    
 7   c_surfactant2       8152 non-null   float64
 8   id_additive1        5504 non-null   str    
 9   smiles_additive1    5504 non-null   str    
 10  c_additive1         8152 non-null   float64
 11  id_additive2        1295 non-null   str    
 12  smiles_additive2    1139 non-null   str    
 13  c_additive2         8152 non-null   float64
 14  temperature         8152 non-null   int64  
 15  viscoelasticity     8152 non-null   float64
 16  g_0              

In [28]:
smiles = visco['smiles_surfactant1']
smiles = pd.concat([smiles, visco['smiles_surfactant2']])
smiles = pd.concat([smiles, visco['smiles_additive1']])
smiles = pd.concat([smiles, visco['smiles_additive2']])
smiles = smiles.drop_duplicates()
smiles = smiles.dropna()
len(smiles)

137

In [37]:
def smiles_to_maccs(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return np.nan

    fp = MACCSkeys.GenMACCSKeys(mol)
    return np.array(fp, dtype=np.uint8)[1:]

In [38]:
smiles = smiles.to_frame(name='smiles')
smiles.head()

,smiles
0,CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+]
8,CCCCCCCCCCCCCCCC[N+](C)(C)CCCS(=O)(=O)[O-]
11,CCCCCCCCCCCCCC[N+](C)(C)CCCS(=O)(=O)[O-]
13,CCCCCCCCCCCCCCCC[N+](C)(C)C.Cc1ccc(S(=O)(=O)[O...
87,CCCCCCCCCCCCOCCOCCOCCOS(=O)(=O)[O-].[Na+]


In [39]:
maccs = pd.DataFrame(
    smiles["smiles"].apply(smiles_to_maccs).tolist(),
    index=smiles.index,
    columns=[f"maccs_{i}" for i in range(1, 167)]
)

maccs = pd.concat([smiles, maccs], axis=1)

In [40]:
maccs['smiles'].head()

0                   CCCCCCCCCCCCOCCOS(=O)(=O)[O-].[Na+]
8            CCCCCCCCCCCCCCCC[N+](C)(C)CCCS(=O)(=O)[O-]
11             CCCCCCCCCCCCCC[N+](C)(C)CCCS(=O)(=O)[O-]
13    CCCCCCCCCCCCCCCC[N+](C)(C)C.Cc1ccc(S(=O)(=O)[O...
87            CCCCCCCCCCCCOCCOCCOCCOS(=O)(=O)[O-].[Na+]
Name: smiles, dtype: str

In [41]:
maccs.to_csv('data/maccs.csv', index=False)